In [1]:
import os
import pandas as pd
from paddleocr import PaddleOCR
import logging
import warnings
import gc


In [2]:
warnings.filterwarnings("ignore")
logging.getLogger('ppocr').setLevel(logging.ERROR)

def extract_street_typography_lts(
    base_dir="E:/street_imagery_project/raw_images", 
    output_csv="E:/street_imagery_project/metadata/extracted_typography.csv"
):
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    
    # 1. Checkpoint Loader
    if os.path.exists(output_csv):
        existing_df = pd.read_csv(output_csv)
        processed_ids = set(existing_df['image_id'].astype(str).unique())
        print(f"Resuming run: Found {len(processed_ids)} already processed images in CSV.")
    else:
        processed_ids = set()
        pd.DataFrame(columns=["neighborhood", "image_id", "detected_text", "confidence_score"]).to_csv(output_csv, index=False, encoding='utf-8-sig')

    print("Initializing OCR Engine (Stable LTS Model on GPU)...")
    # Natively supports the GPU flag and the standard angle classifier
    ocr = PaddleOCR(use_angle_cls=True, lang='korean', use_gpu=True, show_log=False)
    
    neighborhoods = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]

    for neighborhood in neighborhoods:
        print(f"\nScanning {neighborhood.title()}...")
        neighborhood_path = os.path.join(base_dir, neighborhood)
        
        image_paths = [
            os.path.join(neighborhood_path, f) for f in os.listdir(neighborhood_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
        ]
        
        batch_extracted = []
        total_images = len(image_paths)
        
        for i, img_path in enumerate(image_paths, 1):
            image_id = os.path.basename(img_path).replace(".png", "").replace(".jpg", "")
            
            # Skip Mapillary timeouts
            if image_id in processed_ids or os.path.getsize(img_path) < 5000:
                continue

            try:
                result = ocr.ocr(img_path)
                
                # Standard v2.8.1 Parser
                if result and isinstance(result, list) and result[0]:
                    for line in result[0]:
                        if not isinstance(line, (list, tuple)) or len(line) < 2:
                            continue
                            
                        text_data = line[1]
                        if isinstance(text_data, (tuple, list)) and len(text_data) >= 2:
                            detected_text = str(text_data[0])
                            confidence = float(text_data[1])
                        elif isinstance(text_data, str):
                            detected_text = text_data
                            confidence = 0.99 
                        else:
                            continue
                        
                        if confidence > 0.60:
                            batch_extracted.append({
                                "neighborhood": neighborhood,
                                "image_id": image_id,
                                "detected_text": detected_text,
                                "confidence_score": round(confidence, 4)
                            })
            except Exception:
                pass
            
            # Memory flush
            if len(batch_extracted) >= 50:
                pd.DataFrame(batch_extracted).to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8-sig')
                batch_extracted = []
                gc.collect()

            if i % 100 == 0 or i == total_images:
                print(f"   [{i}/{total_images}] images processed in {neighborhood.title()}")

        if batch_extracted:
            pd.DataFrame(batch_extracted).to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8-sig')
            gc.collect()
            
        print(f"Finished {neighborhood.title()} - Progress securely written to disk.")

    print(f"\nAll neighborhoods complete. Final dataset intact at: {output_csv}")

if __name__ == "__main__":
    extract_street_typography_lts()

Resuming run: Found 0 already processed images in CSV.
Initializing OCR Engine (Stable LTS Model on GPU)...
download https://paddleocr.bj.bcebos.com/PP-OCRv3/multilingual/Multilingual_PP-OCRv3_det_infer.tar to C:\Users\barre/.paddleocr/whl\det\ml\Multilingual_PP-OCRv3_det_infer\Multilingual_PP-OCRv3_det_infer.tar


100%|██████████| 3.85M/3.85M [00:11<00:00, 350kiB/s] 


download https://paddleocr.bj.bcebos.com/PP-OCRv4/multilingual/korean_PP-OCRv4_rec_infer.tar to C:\Users\barre/.paddleocr/whl\rec\korean\korean_PP-OCRv4_rec_infer\korean_PP-OCRv4_rec_infer.tar


100%|██████████| 24.4M/24.4M [02:25<00:00, 168kiB/s] 


download https://paddleocr.bj.bcebos.com/dygraph_v2.0/ch/ch_ppocr_mobile_v2.0_cls_infer.tar to C:\Users\barre/.paddleocr/whl\cls\ch_ppocr_mobile_v2.0_cls_infer\ch_ppocr_mobile_v2.0_cls_infer.tar


100%|██████████| 2.19M/2.19M [00:04<00:00, 484kiB/s] 



Scanning Gangnam...
   [100/300] images processed in Gangnam
   [200/300] images processed in Gangnam
   [300/300] images processed in Gangnam
Finished Gangnam - Progress securely written to disk.

Scanning Hongdae...
   [100/300] images processed in Hongdae
   [200/300] images processed in Hongdae
   [300/300] images processed in Hongdae
Finished Hongdae - Progress securely written to disk.

Scanning Itaewon...
   [100/300] images processed in Itaewon
   [200/300] images processed in Itaewon
   [300/300] images processed in Itaewon
Finished Itaewon - Progress securely written to disk.

Scanning Jeju City...
   [100/300] images processed in Jeju City
   [200/300] images processed in Jeju City
   [300/300] images processed in Jeju City
Finished Jeju City - Progress securely written to disk.

Scanning Jongno...
   [100/300] images processed in Jongno
   [200/300] images processed in Jongno
   [300/300] images processed in Jongno
Finished Jongno - Progress securely written to disk.

Scan

In [4]:
import os
import time
import requests
import pandas as pd

def reconstruct_spatial_metadata(
    base_dir="E:/street_imagery_project/raw_images", 
    output_csv="E:/street_imagery_project/metadata/sampled_pids.csv",
    access_token="MLY|26710645935302388|1496476136fe59863a3f5ceda4c38b71"
):
    if access_token == "YOUR_MAPILLARY_TOKEN_HERE":
        print("⚠️ Please insert your Mapillary access token in the script before running.")
        return

    os.makedirs(os.path.dirname(output_csv), exist_ok=True)

    # 1. Checkpoint Loader
    if os.path.exists(output_csv):
        existing_df = pd.read_csv(output_csv)
        # Ensure IDs are strings to match file names
        processed_ids = set(existing_df['image_id'].astype(str).unique())
        print(f"Resuming recovery: Found {len(processed_ids)} already recovered coordinates.")
    else:
        processed_ids = set()
        pd.DataFrame(columns=["image_id", "neighborhood", "longitude", "latitude"]).to_csv(output_csv, index=False, encoding='utf-8-sig')

    # 2. Inventory Local Ground Truth
    local_inventory = []
    neighborhoods = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    
    for neighborhood in neighborhoods:
        neighborhood_path = os.path.join(base_dir, neighborhood)
        for file in os.listdir(neighborhood_path):
            if file.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
                image_id = file.rsplit('.', 1)[0]
                if image_id not in processed_ids:
                    local_inventory.append({"image_id": image_id, "neighborhood": neighborhood})

    total_to_recover = len(local_inventory)
    print(f"Found {total_to_recover} local images requiring coordinate recovery.")

    # 3. Query the Mapillary API
    headers = {"Authorization": f"OAuth {access_token}"}
    batch_recovered = []
    
    for i, item in enumerate(local_inventory, 1):
        image_id = item["image_id"]
        neighborhood = item["neighborhood"]
        url = f"https://graph.mapillary.com/{image_id}?fields=geometry"
        
        try:
            response = requests.get(url, headers=headers)
            
            if response.status_code == 200:
                data = response.json()
                # Mapillary GeoJSON returns [longitude, latitude]
                if 'geometry' in data and data['geometry'] is not None:
                    coords = data['geometry']['coordinates']
                    batch_recovered.append({
                        "image_id": image_id,
                        "neighborhood": neighborhood,
                        "longitude": coords[0],
                        "latitude": coords[1]
                    })
            elif response.status_code == 429:
                # Handle API Rate Limiting gracefully
                print("\nAPI Rate limit hit. Sleeping for 10 seconds...")
                time.sleep(10)
                continue 
            else:
                print(f"Failed to fetch ID {image_id}: HTTP {response.status_code}")

        except Exception as e:
            print(f"Connection error on ID {image_id}: {e}")

        # 4. Checkpoint flush to disk every 50 records
        if len(batch_recovered) >= 50:
            pd.DataFrame(batch_recovered).to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8-sig')
            batch_recovered = []
            # Small sleep to prevent hammering the API
            time.sleep(0.5)

        if i % 100 == 0 or i == total_to_recover:
            print(f"   [{i}/{total_to_recover}] coordinates recovered")
            time.sleep(1) # Extra breather every 100 requests

    # Flush final batch
    if batch_recovered:
        pd.DataFrame(batch_recovered).to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8-sig')

    print(f"\n✅ Recovery complete! Spatial coordinates rebuilt at: {output_csv}")

if __name__ == "__main__":
    reconstruct_spatial_metadata()

Found 2700 local images requiring coordinate recovery.
   [100/2700] coordinates recovered
   [200/2700] coordinates recovered
   [300/2700] coordinates recovered
   [400/2700] coordinates recovered
   [500/2700] coordinates recovered
   [600/2700] coordinates recovered
   [700/2700] coordinates recovered
   [800/2700] coordinates recovered
   [900/2700] coordinates recovered
   [1000/2700] coordinates recovered
   [1100/2700] coordinates recovered
   [1200/2700] coordinates recovered
   [1300/2700] coordinates recovered
   [1400/2700] coordinates recovered
   [1500/2700] coordinates recovered
   [1600/2700] coordinates recovered
   [1700/2700] coordinates recovered
   [1800/2700] coordinates recovered
   [1900/2700] coordinates recovered
   [2000/2700] coordinates recovered
   [2100/2700] coordinates recovered
   [2200/2700] coordinates recovered
   [2300/2700] coordinates recovered
   [2400/2700] coordinates recovered
   [2500/2700] coordinates recovered
   [2600/2700] coordinates re

In [5]:
import pandas as pd
import re
import os

def build_spatial_master():
    gps_csv = "E:/street_imagery_project/metadata/sampled_pids.csv"
    ocr_csv = "E:/street_imagery_project/metadata/extracted_typography.csv"
    output_csv = "E:/street_imagery_project/metadata/spatial_typography_master.csv"

    print("Loading datasets...")
    df_gps = pd.read_csv(gps_csv)
    df_ocr = pd.read_csv(ocr_csv)

    # Cast IDs to string to prevent float mismatch during the merge
    df_gps['image_id'] = df_gps['image_id'].astype(str)
    df_ocr['image_id'] = df_ocr['image_id'].astype(str)

    print("Applying regex language classification...")
    def classify_text(text):
        text_str = str(text)
        # Standard Unicode range for Korean Hangul Syllables and Jamo
        has_hangul = bool(re.search(r'[\u3131-\u318E\uAC00-\uD7A3]', text_str))
        has_english = bool(re.search(r'[a-zA-Z]', text_str))
        
        if has_hangul and has_english:
            return 'Mixed'
        elif has_hangul:
            return 'Hangul'
        elif has_english:
            return 'English'
        return 'Other'

    df_ocr['language'] = df_ocr['detected_text'].apply(classify_text)

    print("Aggregating typography to the spatial frame level...")
    def determine_frame_language(langs):
        langs_set = set(langs)
        if 'Mixed' in langs_set or ('Hangul' in langs_set and 'English' in langs_set):
            return 'Mixed'
        elif 'Hangul' in langs_set:
            return 'Hangul'
        elif 'English' in langs_set:
            return 'English'
        elif 'Other' in langs_set:
            return 'Other'
        return 'No_Text'

    image_agg = df_ocr.groupby('image_id').agg(
        sign_count=('detected_text', 'count'),
        frame_language=('language', determine_frame_language),
        combined_text=('detected_text', lambda x: ' | '.join(x.astype(str)))
    ).reset_index()

    print("Merging with GPS coordinates to preserve zero-text baseline...")
    master_df = pd.merge(df_gps, image_agg, on='image_id', how='left')

    # Fill NaNs for the zero-sign baseline frames
    master_df['sign_count'] = master_df['sign_count'].fillna(0).astype(int)
    master_df['frame_language'] = master_df['frame_language'].fillna('No_Text')
    master_df['combined_text'] = master_df['combined_text'].fillna('None')

    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    master_df.to_csv(output_csv, index=False, encoding='utf-8-sig')

    print(f"\n✅ Success! Master dataset compiled.")
    print(f"Total spatial points: {len(master_df)}")
    print("\nLanguage Distribution:")
    print(master_df['frame_language'].value_counts())
    
    return master_df

if __name__ == "__main__":
    df_master = build_spatial_master()

Loading datasets...
Applying regex language classification...
Aggregating typography to the spatial frame level...
Merging with GPS coordinates to preserve zero-text baseline...

✅ Success! Master dataset compiled.
Total spatial points: 2700

Language Distribution:
frame_language
No_Text    2700
Name: count, dtype: int64


In [3]:
import pandas as pd
import re
import os

def build_spatial_master():
    gps_csv = "E:/street_imagery_project/metadata/sampled_pids.csv"
    ocr_csv = "E:/street_imagery_project/metadata/extracted_typography.csv"
    output_csv = "E:/street_imagery_project/metadata/spatial_typography_master.csv"

    print("Loading datasets...")
    df_gps = pd.read_csv(gps_csv, dtype={'image_id': str})
    df_ocr = pd.read_csv(ocr_csv, dtype={'image_id': str})

    # --- THE FIX: AGGRESSIVE EXTENSION SCRUBBING ---
    # This strips out .jpg, .jpeg, .png, .webp (case insensitive) from the IDs
    df_gps['image_id'] = df_gps['image_id'].str.replace(r'\.(jpg|jpeg|png|webp)$', '', flags=re.IGNORECASE, regex=True).str.strip()
    df_ocr['image_id'] = df_ocr['image_id'].str.replace(r'\.(jpg|jpeg|png|webp)$', '', flags=re.IGNORECASE, regex=True).str.strip()

    print("Applying regex language classification...")
    def classify_text(text):
        text_str = str(text)
        has_hangul = bool(re.search(r'[\u3131-\u318E\uAC00-\uD7A3]', text_str))
        has_english = bool(re.search(r'[a-zA-Z]', text_str))
        
        if has_hangul and has_english:
            return 'Mixed'
        elif has_hangul:
            return 'Hangul'
        elif has_english:
            return 'English'
        return 'Other'

    df_ocr['language'] = df_ocr['detected_text'].apply(classify_text)

    print("Aggregating typography to the spatial frame level...")
    def determine_frame_language(langs):
        langs_set = set(langs)
        if 'Mixed' in langs_set or ('Hangul' in langs_set and 'English' in langs_set):
            return 'Mixed'
        elif 'Hangul' in langs_set:
            return 'Hangul'
        elif 'English' in langs_set:
            return 'English'
        elif 'Other' in langs_set:
            return 'Other'
        return 'No_Text'

    image_agg = df_ocr.groupby('image_id').agg(
        sign_count=('detected_text', 'count'),
        frame_language=('language', determine_frame_language),
        combined_text=('detected_text', lambda x: ' | '.join(x.astype(str)))
    ).reset_index()

    print("Merging with GPS coordinates to preserve zero-text baseline...")
    master_df = pd.merge(df_gps, image_agg, on='image_id', how='left')

    # Fill NaNs for the zero-sign baseline frames
    master_df['sign_count'] = master_df['sign_count'].fillna(0).astype(int)
    master_df['frame_language'] = master_df['frame_language'].fillna('No_Text')
    master_df['combined_text'] = master_df['combined_text'].fillna('None')

    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    master_df.to_csv(output_csv, index=False, encoding='utf-8-sig')

    print(f"\n✅ Success! Master dataset compiled.")
    print(f"Total spatial points: {len(master_df)}")
    print("\nLanguage Distribution:")
    print(master_df['frame_language'].value_counts())
    
    return master_df

if __name__ == "__main__":
    df_master = build_spatial_master()

Loading datasets...
Applying regex language classification...
Aggregating typography to the spatial frame level...
Merging with GPS coordinates to preserve zero-text baseline...

✅ Success! Master dataset compiled.
Total spatial points: 2700

Language Distribution:
frame_language
No_Text    2700
Name: count, dtype: int64


In [2]:
import pandas as pd

gps_csv = "E:/street_imagery_project/metadata/sampled_pids.csv"
ocr_csv = "E:/street_imagery_project/metadata/extracted_typography.csv"

# Load as raw strings
df_gps = pd.read_csv(gps_csv, dtype={'image_id': str})
df_ocr = pd.read_csv(ocr_csv, dtype={'image_id': str})

print("--- DATA PIPELINE DIAGNOSTIC ---")
print(f"GPS Dataset: {len(df_gps)} rows.")
print(f"GPS Sample IDs: {df_gps['image_id'].head(3).tolist()}")
print("-" * 30)

print(f"OCR Dataset: {len(df_ocr)} rows.")
if len(df_ocr) > 0:
    print(f"OCR Sample IDs: {df_ocr['image_id'].head(3).tolist()}")
    
    # Strip extensions just like our script did to test the match
    import re
    gps_clean = set(df_gps['image_id'].str.replace(r'\.(jpg|jpeg|png|webp)$', '', flags=re.IGNORECASE, regex=True).str.strip())
    ocr_clean = set(df_ocr['image_id'].str.replace(r'\.(jpg|jpeg|png|webp)$', '', flags=re.IGNORECASE, regex=True).str.strip())
    
    overlap = gps_clean.intersection(ocr_clean)
    print("-" * 30)
    print(f"Successful Matches found: {len(overlap)}")
else:
    print("⚠️ YOUR OCR DATASET IS EMPTY. The OCR script didn't save any text.")

--- DATA PIPELINE DIAGNOSTIC ---
GPS Dataset: 2700 rows.
GPS Sample IDs: ['1003197857963721', '1017147865355803', '1033995711879804']
------------------------------
OCR Dataset: 0 rows.
⚠️ YOUR OCR DATASET IS EMPTY. The OCR script didn't save any text.


In [1]:
import os
from paddleocr import PaddleOCR

def test_single_image():
    base_dir = "E:/street_imagery_project/raw_images"
    
    # 1. Find exactly one image to test
    neighborhoods = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]
    test_image = None
    
    for n in neighborhoods:
        n_path = os.path.join(base_dir, n)
        files = [f for f in os.listdir(n_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
        if files:
            test_image = os.path.join(n_path, files[0])
            break
            
    if not test_image:
        print("No images found in your raw_images folders!")
        return
        
    print(f"Testing OCR on: {test_image}")
    print("Initializing engine...")
    
    # 2. Initialize with logging turned ON
    ocr = PaddleOCR(use_angle_cls=True, lang='korean', use_gpu=False, enable_mkldnn=True, show_log=False)
    
    print("Running extraction (No error suppression)...")
    # 3. Run without try/except so it crashes loudly
    result = ocr.ocr(test_image)
    
    print("\n--- RAW OCR OUTPUT ---")
    print(result)

if __name__ == "__main__":
    test_single_image()

Testing OCR on: E:/street_imagery_project/raw_images\Gangnam\1003197857963721.png
Initializing engine...
Running extraction (No error suppression)...

--- RAW OCR OUTPUT ---
[[[[[833.0, 307.0], [869.0, 307.0], [869.0, 322.0], [833.0, 322.0]], ('출과', 0.6748137474060059)], [[[837.0, 319.0], [873.0, 323.0], [872.0, 335.0], [836.0, 331.0]], ('듬엄B', 0.5431529879570007)], [[[985.0, 322.0], [1019.0, 320.0], [1020.0, 331.0], [986.0, 334.0]], ('2412', 0.7709975242614746)], [[[991.0, 335.0], [1022.0, 335.0], [1022.0, 347.0], [991.0, 347.0]], ('다어아든', 0.7903577089309692)]]]


In [1]:
import os
import pandas as pd
from paddleocr import PaddleOCR
import logging
import warnings
import gc

warnings.filterwarnings("ignore")
logging.getLogger('ppocr').setLevel(logging.ERROR)

def extract_street_typography_cpu(
    base_dir="E:/street_imagery_project/raw_images", 
    output_csv="E:/street_imagery_project/metadata/extracted_typography.csv"
):
    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    
    # Checkpoint Loader
    if os.path.exists(output_csv):
        existing_df = pd.read_csv(output_csv)
        processed_ids = set(existing_df['image_id'].astype(str).unique())
        print(f"Resuming run: Found {len(processed_ids)} already processed images in CSV.")
    else:
        processed_ids = set()
        pd.DataFrame(columns=["neighborhood", "image_id", "detected_text", "confidence_score"]).to_csv(output_csv, index=False, encoding='utf-8-sig')

    print("Initializing OCR Engine (CPU mode with Intel MKLDNN optimization)...")
    # --- THE FIX: GPU DISABLED, MKLDNN ENABLED ---
    ocr = PaddleOCR(use_angle_cls=True, lang='korean', use_gpu=False, enable_mkldnn=True, show_log=False)
    
    neighborhoods = [d for d in os.listdir(base_dir) if os.path.isdir(os.path.join(base_dir, d))]

    for neighborhood in neighborhoods:
        print(f"\nScanning {neighborhood.title()}...")
        neighborhood_path = os.path.join(base_dir, neighborhood)
        
        image_paths = [
            os.path.join(neighborhood_path, f) for f in os.listdir(neighborhood_path)
            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))
        ]
        
        batch_extracted = []
        total_images = len(image_paths)
        
        for i, img_path in enumerate(image_paths, 1):
            image_id = os.path.basename(img_path).replace(".png", "").replace(".jpg", "").replace(".jpeg", "").replace(".webp", "")
            
            if image_id in processed_ids or os.path.getsize(img_path) < 5000:
                continue

            try:
                result = ocr.ocr(img_path)
                
                if result and isinstance(result, list) and result[0]:
                    for line in result[0]:
                        if not isinstance(line, (list, tuple)) or len(line) < 2:
                            continue
                            
                        text_data = line[1]
                        if isinstance(text_data, (tuple, list)) and len(text_data) >= 2:
                            detected_text = str(text_data[0])
                            confidence = float(text_data[1])
                        elif isinstance(text_data, str):
                            detected_text = text_data
                            confidence = 0.99 
                        else:
                            continue
                        
                        if confidence > 0.50:
                            batch_extracted.append({
                                "neighborhood": neighborhood,
                                "image_id": image_id,
                                "detected_text": detected_text,
                                "confidence_score": round(confidence, 4)
                            })
            except Exception:
                pass
            
            # Flush to disk aggressively
            if len(batch_extracted) >= 50:
                pd.DataFrame(batch_extracted).to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8-sig')
                batch_extracted = []
                gc.collect()

            if i % 100 == 0 or i == total_images:
                print(f"   [{i}/{total_images}] images processed in {neighborhood.title()}")

        if batch_extracted:
            pd.DataFrame(batch_extracted).to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8-sig')
            gc.collect()
            
        print(f"Finished {neighborhood.title()} - Progress securely written to disk.")

    print(f"\n✅ All neighborhoods complete. Final text dataset saved at: {output_csv}")

if __name__ == "__main__":
    extract_street_typography_cpu()

Initializing OCR Engine (CPU mode with Intel MKLDNN optimization)...

Scanning Gangnam...
   [100/300] images processed in Gangnam
   [200/300] images processed in Gangnam
   [300/300] images processed in Gangnam
Finished Gangnam - Progress securely written to disk.

Scanning Hongdae...
   [100/300] images processed in Hongdae
   [200/300] images processed in Hongdae
   [300/300] images processed in Hongdae
Finished Hongdae - Progress securely written to disk.

Scanning Itaewon...
   [100/300] images processed in Itaewon
   [200/300] images processed in Itaewon
   [300/300] images processed in Itaewon
Finished Itaewon - Progress securely written to disk.

Scanning Jeju City...
   [100/300] images processed in Jeju City
   [200/300] images processed in Jeju City
   [300/300] images processed in Jeju City
Finished Jeju City - Progress securely written to disk.

Scanning Jongno...
   [100/300] images processed in Jongno
   [200/300] images processed in Jongno
   [300/300] images processed

In [2]:
import pandas as pd
import re
import os

def build_spatial_master():
    gps_csv = "E:/street_imagery_project/metadata/sampled_pids.csv"
    ocr_csv = "E:/street_imagery_project/metadata/extracted_typography.csv"
    output_csv = "E:/street_imagery_project/metadata/spatial_typography_master.csv"

    print("Loading datasets...")
    df_gps = pd.read_csv(gps_csv, dtype={'image_id': str})
    df_ocr = pd.read_csv(ocr_csv, dtype={'image_id': str})

    # --- THE FIX: AGGRESSIVE EXTENSION SCRUBBING ---
    # This strips out .jpg, .jpeg, .png, .webp (case insensitive) from the IDs
    df_gps['image_id'] = df_gps['image_id'].str.replace(r'\.(jpg|jpeg|png|webp)$', '', flags=re.IGNORECASE, regex=True).str.strip()
    df_ocr['image_id'] = df_ocr['image_id'].str.replace(r'\.(jpg|jpeg|png|webp)$', '', flags=re.IGNORECASE, regex=True).str.strip()

    print("Applying regex language classification...")
    def classify_text(text):
        text_str = str(text)
        has_hangul = bool(re.search(r'[\u3131-\u318E\uAC00-\uD7A3]', text_str))
        has_english = bool(re.search(r'[a-zA-Z]', text_str))
        
        if has_hangul and has_english:
            return 'Mixed'
        elif has_hangul:
            return 'Hangul'
        elif has_english:
            return 'English'
        return 'Other'

    df_ocr['language'] = df_ocr['detected_text'].apply(classify_text)

    print("Aggregating typography to the spatial frame level...")
    def determine_frame_language(langs):
        langs_set = set(langs)
        if 'Mixed' in langs_set or ('Hangul' in langs_set and 'English' in langs_set):
            return 'Mixed'
        elif 'Hangul' in langs_set:
            return 'Hangul'
        elif 'English' in langs_set:
            return 'English'
        elif 'Other' in langs_set:
            return 'Other'
        return 'No_Text'

    image_agg = df_ocr.groupby('image_id').agg(
        sign_count=('detected_text', 'count'),
        frame_language=('language', determine_frame_language),
        combined_text=('detected_text', lambda x: ' | '.join(x.astype(str)))
    ).reset_index()

    print("Merging with GPS coordinates to preserve zero-text baseline...")
    master_df = pd.merge(df_gps, image_agg, on='image_id', how='left')

    # Fill NaNs for the zero-sign baseline frames
    master_df['sign_count'] = master_df['sign_count'].fillna(0).astype(int)
    master_df['frame_language'] = master_df['frame_language'].fillna('No_Text')
    master_df['combined_text'] = master_df['combined_text'].fillna('None')

    os.makedirs(os.path.dirname(output_csv), exist_ok=True)
    master_df.to_csv(output_csv, index=False, encoding='utf-8-sig')

    print(f"\n✅ Success! Master dataset compiled.")
    print(f"Total spatial points: {len(master_df)}")
    print("\nLanguage Distribution:")
    print(master_df['frame_language'].value_counts())
    
    return master_df

if __name__ == "__main__":
    df_master = build_spatial_master()

Loading datasets...
Applying regex language classification...
Aggregating typography to the spatial frame level...
Merging with GPS coordinates to preserve zero-text baseline...

✅ Success! Master dataset compiled.
Total spatial points: 2700

Language Distribution:
frame_language
No_Text    1467
Hangul      499
Mixed       492
English     168
Other        74
Name: count, dtype: int64
